In [24]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from plotly.subplots import make_subplots

In [25]:
df = pd.read_csv('data/event_data.csv')

In [26]:
df.head()

,event_date,event_time,event_type,product_id,category_id,category_code,price,user_id,user_session
0,2020-03-01,2020-03-01 00:00:19 UTC,view,3600145,2232732092297380188,appliances.kitchen.washer,179.71,621538299,189da4a9-76c0-4ede-8d64-cfdf1a3bb76c
1,2020-03-01,2020-03-01 00:00:59 UTC,view,3601248,2232732092297380188,appliances.kitchen.washer,181.96,621538299,189da4a9-76c0-4ede-8d64-cfdf1a3bb76c
2,2020-03-01,2020-03-01 00:01:58 UTC,view,100068488,2232732093077520756,construction.tools.light,293.13,519011545,b6e003a3-c13b-4193-853b-f17f0752c5a9
3,2020-03-01,2020-03-01 00:03:16 UTC,view,3601248,2232732092297380188,appliances.kitchen.washer,181.96,621538299,189da4a9-76c0-4ede-8d64-cfdf1a3bb76c
4,2020-03-01,2020-03-01 00:03:21 UTC,view,5300405,2232732089269092627,NaN,17.99,572843444,bb10c16f-36df-4537-8fb2-b8eb22fec92b


In [42]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 10 columns):
 #   Column         Non-Null Count    Dtype                        
---  ------         --------------    -----                        
 0   event_time     1048575 non-null  datetime64[ns, Europe/Moscow]
 1   event_type     1048575 non-null  object                       
 2   product_id     1048575 non-null  int64                        
 3   category_id    1048575 non-null  int64                        
 4   category_code  939415 non-null   category                     
 5   price          1048575 non-null  float64                      
 6   user_id        1048575 non-null  int64                        
 7   user_session   1048575 non-null  category                     
 8   day_of_week    1048575 non-null  int32                        
 9   hour           1048575 non-null  int32                        
dtypes: category(2), datetime64[ns, Europe/Moscow](1), float64(1), int3

In [ ]:
df['event_time'] = pd.to_datetime(df['event_time'], utc=True)
df['event_time'] = df['event_time'].dt.tz_convert('Europe/Moscow')
df['user_session'] = df['user_session'].astype('category')
df['category_code'] = df['category_code'].astype('category')
df.drop(columns=['event_date'], inplace=True)

## Funnels

In [29]:
# 1. сделать график воронку по разному времени дня

result = df[['event_time', 'event_type', 'user_id']].copy()
result['hour'] = result['event_time'].dt.hour          

conditions = [
    (result['hour'] >= 6)  & (result['hour'] < 12),
    (result['hour'] >= 12) & (result['hour'] < 18),
    (result['hour'] >= 18) & (result['hour'] < 23),
]
choices = ['morning', 'day', 'evening']
result['day_part'] = np.select(conditions, choices, default='night')

# Задаём порядок 
order_event = ['view', 'cart', 'purchase']
order_day = ['morning', 'day', 'evening', 'night']

result['event_type'] = pd.Categorical(result['event_type'], categories=order_event, ordered=True)
result['day_part']   = pd.Categorical(result['day_part'],   categories=order_day,   ordered=True)

# Группировка
g = result.groupby(['day_part', 'event_type'])['user_id'].nunique().reset_index()
g = g.sort_values(['day_part', 'event_type'])

g['conv_prev'] = (
    g.groupby('day_part', observed=True)['user_id']
    .transform(lambda x: (x / x.shift(1) * 100).round(1))
)
g['conv_prev'] = g['conv_prev'].fillna(100)
g


/var/folders/w5/njx7ldwn63j1v74_7qjmpfnc0000gn/T/ipykernel_82277/2051594355.py:22: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



,day_part,event_type,user_id,conv_prev
0,morning,view,41609,100.0
1,morning,cart,7344,17.7
2,morning,purchase,4167,56.7
3,day,view,42446,100.0
4,day,cart,7522,17.7
5,day,purchase,4232,56.3
6,evening,view,30534,100.0
7,evening,cart,4650,15.2
8,evening,purchase,2309,49.7
9,night,view,12943,100.0


In [30]:
fig = px.funnel(
    g,
    x='user_id',
    y='event_type',
    facet_col='day_part',
    facet_col_wrap=2,
    title='Утром покупают охотнее 🥳',
    text='conv_prev'
)
fig.update_traces(texttemplate='%{text}%')
fig.show()

In [31]:
# 2. Cделать воронки для разных типов покупателей 
g = pd.crosstab(df['user_id'], df['event_type'])
g = g.sort_values('purchase', ascending=False)
g['user_category'] = pd.cut(g['purchase'], bins=[-1, 1, 10, float('inf')],
                              labels=['new', 'regular', 'loyal'])
g = g[['view', 'cart', 'purchase', 'user_category']]                              
g

event_type,view,cart,purchase,user_category
user_id,,,,
513230794,541,273,238,loyal
547220387,224,128,88,loyal
560837658,196,145,82,loyal
616286657,210,82,75,loyal
577149809,305,75,72,loyal
...,...,...,...,...
561773656,4,0,0,new
561772998,3,0,0,new
561770924,61,8,0,new


In [32]:
# Суммируем действия по категориям пользователей
funnel_data = (
    g.groupby('user_category')[['view', 'cart', 'purchase']]
    .sum()
    .reindex(columns=['view', 'cart', 'purchase'])  # порядок стадий
    .reset_index()
    .melt(id_vars='user_category', var_name='event_type', value_name='count') # сворачиваем юзеров по категориям
)

funnel_data['conv_prev'] = (
    funnel_data
    .groupby('user_category', observed=True)['count']
    .transform(lambda x: (x / x.shift(1) * 100).round(1))
)
funnel_data['conv_prev'] = funnel_data['conv_prev'].fillna(100)
funnel_data

fig = px.funnel(
    funnel_data,
    x='count',          # ширина бара = реальное количество событий
    y='event_type',
    facet_col='user_category',   # все категории в ОДНОМ графике
    text='conv_prev',
    title='Конверсия старичков намного выше'
)
fig.update_traces(texttemplate='%{text}%')
fig.show()

/var/folders/w5/njx7ldwn63j1v74_7qjmpfnc0000gn/T/ipykernel_82277/3397938894.py:3: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



## Metrics

In [33]:
# 1. Пиковое время активности: в какое время суток/день недели больше всего просмотров и покупок
df['day_of_week'] = df['event_time'].dt.dayofweek
df['hour'] = df['event_time'].dt.hour 


In [34]:
sales = (
    df[df['event_type'] == 'purchase']
    .groupby(['day_of_week', 'hour'])['user_id']
    .count()
    .reset_index()
)

day_names = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
sales['day_of_week'] = sales['day_of_week'].map(day_names)
sales.rename(columns={'user_id': 'sales'}, inplace=True) 

sales.head(5)


,day_of_week,hour,sales
0,Monday,0,29
1,Monday,1,24
2,Monday,2,21
3,Monday,3,27
4,Monday,4,29


In [35]:
fig = px.bar(
    sales, 
    x='hour', 
    y='sales',
    facet_col='day_of_week',
    facet_col_wrap=3,
    title='Продажи по часам и дням недели',
    facet_row_spacing=0.13,
)
fig.update_xaxes(showticklabels=True)
fig.show()

### Вывод: Пик продаж в понедельник в 9 утра по Москве

# Продолжаем

In [53]:
# 2. Процент сессий, не закончившихся оплатой собранной корзины (~процент брошенных корзин)
sales = (
    df.groupby('event_type')['category_id']
    .count()
    .reset_index()
)
sales.sort_values('category_id', ascending=False, inplace=True, ignore_index=True)
sales['category_id'] = sales['category_id'].astype('int')
sales.head(3)

,event_type,category_id
0,view,972713
1,cart,55911
2,purchase,19951


In [60]:
ans = 100 - (sales.loc[2, 'category_id'] / sales.loc[1, 'category_id'] * 100)
print(f'Брошенные корзины составляют: {int(ans)}%')

Брошенные корзины составляют: 64%


In [63]:
df.head(3)

,event_time,event_type,product_id,category_id,category_code,price,user_id,user_session,day_of_week,hour
0,2020-03-01 03:00:19+03:00,view,3600145,2232732092297380188,appliances.kitchen.washer,179.71,621538299,189da4a9-76c0-4ede-8d64-cfdf1a3bb76c,6,3
1,2020-03-01 03:00:59+03:00,view,3601248,2232732092297380188,appliances.kitchen.washer,181.96,621538299,189da4a9-76c0-4ede-8d64-cfdf1a3bb76c,6,3
2,2020-03-01 03:01:58+03:00,view,100068488,2232732093077520756,construction.tools.light,293.13,519011545,b6e003a3-c13b-4193-853b-f17f0752c5a9,6,3


In [85]:
# 3. Среднее время от первого просмотра до первой покупки (time to purchase)
first_view = (
    df[df['event_type'] == 'view']
    .groupby('user_session', observed=True)['event_time']
    .min()
)

first_purchase = (
    df[df['event_type'] == 'purchase']
    .groupby('user_session', observed=True)['event_time']
    .min()
)

time_to_purchase = (first_purchase - first_view).dropna()
mean_time = time_to_purchase.mean()
minutes = int(mean_time.total_seconds() / 60)
seconds = int(mean_time.total_seconds() % 60)
print(f'Среднее время от первого просмотра до первой покупки: {minutes} минут {seconds} секунд')


Среднее время от первого просмотра до первой покупки: 7 минут 55 секунд


In [ ]:
# 4. Средняя стоимость покупки (average order value, AOV) 
# и средняя стоимость купленного товара (average item value, AIV)




In [ ]:
# В ответе укажите метрики и полученные значения. Насколько фактические данные совпали с вашими ожиданиями/опытом? 
# Какие интересные наблюдения вы извлекли?